## Goal
Build a speech intelligence pipeline that:
- takes lecture or meeting audio as input
- transcribes speech to text using Whisper
- produces:
  - a baseline summary
  - improved structured insights
  - action items
  - keywords

## Project Pipeline
Audio File → Whisper → Transcript → Analysis → Summary / Action Items / Keywords

In [2]:
!pip -q install transformers datasets accelerate sentencepiece librosa soundfile pandas
!apt-get -qq install ffmpeg

In [3]:
#3-Imports and project folders
import os
import json
import time
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import librosa
from google.colab import files
from transformers import pipeline

# Create folders
BASE_DIR = Path("/content/speech_project")
AUDIO_DIR = BASE_DIR / "audio"
OUTPUT_DIR = BASE_DIR / "outputs"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folders created:")
print("AUDIO_DIR =", AUDIO_DIR)
print("OUTPUT_DIR =", OUTPUT_DIR)

Project folders created:
AUDIO_DIR = /content/speech_project/audio
OUTPUT_DIR = /content/speech_project/outputs


Checked up to here

In [4]:
#4 - Upload audio files
uploaded = files.upload()

for filename in uploaded.keys():
    destination = AUDIO_DIR / filename
    with open(destination, "wb") as f:
        f.write(uploaded[filename])

print("\nFiles ")
for f in sorted(AUDIO_DIR.iterdir()):
    print("-", f.name)

Saving Clean audio.m4a to Clean audio (1).m4a
Saving Fast audio.m4a to Fast audio (1).m4a
Saving meeting-clip1.mp3 to meeting-clip1 (1).mp3
Saving meeting-clip2.wav to meeting-clip2 (1).wav
Saving Noisy audio.m4a to Noisy audio (1).m4a

Saved files:
- Clean audio (1).m4a
- Fast audio (1).m4a
- Noisy audio (1).m4a
- meeting-clip1 (1).mp3
- meeting-clip2 (1).wav


In [5]:
model_name = "openai/whisper-small"

asr = pipeline(
    task="automatic-speech-recognition",
    model=model_name,
    chunk_length_s=30,
    return_timestamps=False
)

print(f"Loaded model: {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Loaded model: openai/whisper-small


In [6]:
# 6 - summary and extraction functions

STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "is", "are", "was", "were", "be", "to",
    "of", "in", "on", "for", "with", "that", "this", "it", "as", "at", "by", "from",
    "we", "you", "they", "he", "she", "i", "our", "their", "his", "her", "have", "has",
    "had", "do", "does", "did", "will", "would", "can", "could", "should", "about",
    "today", "also", "just", "really", "very", "let", "lets"
}

ACTION_HINTS = [
    "need to", "should", "must", "follow up", "send", "submit",
    "complete", "finish", "review", "prepare", "schedule", "update",
    "will", "please", "can you", "make sure", "handle", "create"
]

def split_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]

def baseline_summary(transcript_text, max_sentences=3):
    sentences = [s.strip() for s in transcript_text.split(".") if s.strip()]
    if not sentences:
        return "No summary available."
    return ". ".join(sentences[:max_sentences]) + "."

def extract_keywords(text, top_k=8):
    words = re.findall(r"\b[a-zA-Z][a-zA-Z\-]+\b", text.lower())
    filtered = [w for w in words if w not in STOPWORDS and len(w) > 2]
    counts = Counter(filtered)
    return [word for word, _ in counts.most_common(top_k)]

def extract_action_items(text):
    sentences = split_sentences(text)
    actions = []
    for s in sentences:
        s_lower = s.lower()
        if any(hint in s_lower for hint in ACTION_HINTS):
            actions.append(s)
    return actions[:5] if actions else ["No clear action items detected."]

def improved_summary(text, max_sentences=4):
    sentences = split_sentences(text)
    if not sentences:
        return "No summary available."
    return " ".join(sentences[:max_sentences])

In [7]:
#7-Process all audio
audio_files = sorted([f for f in AUDIO_DIR.iterdir() if f.is_file()])

if not audio_files:
    raise FileNotFoundError("No audio files found in AUDIO_DIR.")

results_list = []

print(f"Found {len(audio_files)} audio files.\n")

for audio_file in audio_files:
    print("=" * 70)
    print("Processing:", audio_file.name)

    # Load audio
    audio_array, sampling_rate = librosa.load(str(audio_file), sr=16000)

    # Transcribe
    start_time = time.time()
    result = asr({"array": audio_array, "sampling_rate": sampling_rate})
    latency = time.time() - start_time

    transcript = result["text"].strip()

    # Generate outputs
    baseline = baseline_summary(transcript)

    structured_output = {
        "summary": improved_summary(transcript),
        "action_items": extract_action_items(transcript),
        "keywords": extract_keywords(transcript, top_k=8)
    }

    # Print preview
    print("\nTranscript:\n")
    print(transcript)

    print("\nBaseline Summary:\n")
    print(baseline)

    print("\nImproved Summary:\n")
    print(structured_output["summary"])

    print("\nAction Items:")
    for i, item in enumerate(structured_output["action_items"], 1):
        print(f"{i}. {item}")

    print("\nKeywords:")
    print(", ".join(structured_output["keywords"]))

    print(f"\nLatency: {latency:.2f} seconds")

    # Save one record
    record = {
        "audio_file": audio_file.name,
        "transcript": transcript,
        "baseline_summary": baseline,
        "improved_summary": structured_output["summary"],
        "action_items": "; ".join(structured_output["action_items"]),
        "keywords": ", ".join(structured_output["keywords"]),
        "latency_seconds": round(latency, 2)
    }

    results_list.append(record)

    # Save per-file outputs
    json_path = OUTPUT_DIR / f"{audio_file.stem}_results.json"
    txt_path = OUTPUT_DIR / f"{audio_file.stem}_transcript.txt"

    with open(json_path, "w") as f:
        json.dump(record, f, indent=2)

    with open(txt_path, "w") as f:
        f.write(transcript)

print("\nFinished processing all files.")

Found 5 audio files.

Processing: Clean audio (1).m4a


/tmp/ipykernel_2305/2643587008.py:16: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(str(audio_file), sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.


Transcript:

Today we are going to discuss the basics of machine learning and how it is used in real-world applications. Machine learning is a subset of artificial intelligence that allows systems to learn patterns from data and make predictions. There are three main types of machine learning, supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data, while unsupervised learning identifies hidden patterns without labels. These techniques are widely used in areas such as healthcare, finance, and recommendation systems. Understanding these fundamentals is important for building intelligent systems.

Baseline Summary:

Today we are going to discuss the basics of machine learning and how it is used in real-world applications. Machine learning is a subset of artificial intelligence that allows systems to learn patterns from data and make predictions. There are three main types of machine learning, supervised learning, unsupervised learnin

/tmp/ipykernel_2305/2643587008.py:16: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(str(audio_file), sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



Transcript:

Today we are going to discuss the basics of machine learning and how it is used in real world applications. Machine learning is a subset of artificial intelligence that allows systems to learn patterns from data and predictions. There are three types of machine learning, supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data, while unsupervised learning identifies hidden patterns without labels. These techniques are widely used in areas such as healthcare, finance, and recommendation systems. Understanding these fundamentals is important for building intelligent systems.

Baseline Summary:

Today we are going to discuss the basics of machine learning and how it is used in real world applications. Machine learning is a subset of artificial intelligence that allows systems to learn patterns from data and predictions. There are three types of machine learning, supervised learning, unsupervised learning, and reinforcement

/tmp/ipykernel_2305/2643587008.py:16: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(str(audio_file), sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



Transcript:

Today we are going to learn how to discuss the basics of machine learning. We use the real world applications. Machine learning is a sense of artificial intelligence that allows us to understand how to use the data we have, the pictures, the feeling of the machine learning, supervised learning, unsupervised learning, and the enforcement of it. Supervised learning uses the data of our unsupervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses the data of our unsupervised learning to identify sleeping patterns without labels. These techniques are used in areas such as healthcare, finance, and rehabilitation systems. I'm considering these fundamentals as important for working with intelligence systems.

Baseline Summary:

Today we are going to learn how to discuss the basics of machine learning. We use the real world applications. Machine learning is a sense of artificial intelligence that allows us to understand how to use the data we

In [8]:
#8 - combined results table
df_results = pd.DataFrame(results_list)
df_results

,audio_file,transcript,baseline_summary,improved_summary,action_items,keywords,latency_seconds
0,Clean audio (1).m4a,Today we are going to discuss the basics of ma...,Today we are going to discuss the basics of ma...,Today we are going to discuss the basics of ma...,No clear action items detected.,"learning, machine, systems, used, patterns, da...",5.91
1,Fast audio (1).m4a,Today we are going to discuss the basics of ma...,Today we are going to discuss the basics of ma...,Today we are going to discuss the basics of ma...,No clear action items detected.,"learning, machine, systems, used, patterns, da...",2.59
2,Noisy audio (1).m4a,Today we are going to learn how to discuss the...,Today we are going to learn how to discuss the...,Today we are going to learn how to discuss the...,No clear action items detected.,"learning, unsupervised, machine, data, supervi...",3.75
3,meeting-clip1 (1).mp3,"Felly, mae'n gweithio'r gweithio'r gweithio'r ...","Felly, mae'n gweithio'r gweithio'r gweithio'r ...","Felly, mae'n gweithio'r gweithio'r gweithio'r ...",No clear action items detected.,"gweithio, bwysig, felly, mae, wneud, oweithio,...",36.87
4,meeting-clip2 (1).wav,"Well, from my point of view, what Paul is prop...","Well, from my point of view, what Paul is prop...","Well, from my point of view, what Paul is prop...",No clear action items detected.,"paul, sounds, hours, well, point, view, what, ...",1.93


In [9]:
#9 - combined results CSV
csv_path = OUTPUT_DIR / "allResults.csv"
df_results.to_csv(csv_path, index=False)

print("Saved ")
print("-", csv_path)

Saved 
- /content/speech_project/outputs/allResults.csv


In [10]:
# 10. Create evaluation template
evaluation_data = [
    {
        "clip": "Clean audio",
        "transcript_quality": "",
        "baseline_summary_quality": "",
        "improved_summary_quality": "",
        "action_item_quality": "",
        "keyword_quality": "",
        "latency_seconds": "",
        "notes": ""
    },
    {
        "clip": "Fast audio",
        "transcript_quality": "",
        "baseline_summary_quality": "",
        "improved_summary_quality": "",
        "action_item_quality": "",
        "keyword_quality": "",
        "latency_seconds": "",
        "notes": ""
    },
    {
        "clip": "Noisy audio",
        "transcript_quality": "",
        "baseline_summary_quality": "",
        "improved_summary_quality": "",
        "action_item_quality": "",
        "keyword_quality": "",
        "latency_seconds": "",
        "notes": ""
    },
    {
        "clip": "meeting-clip1",
        "transcript_quality": "",
        "baseline_summary_quality": "",
        "improved_summary_quality": "",
        "action_item_quality": "",
        "keyword_quality": "",
        "latency_seconds": "",
        "notes": ""
    },
    {
        "clip": "meeting-clip2",
        "transcript_quality": "",
        "baseline_summary_quality": "",
        "improved_summary_quality": "",
        "action_item_quality": "",
        "keyword_quality": "",
        "latency_seconds": "",
        "notes": ""
    }
]

df_eval = pd.DataFrame(evaluation_data)
df_eval

,clip,transcript_quality,baseline_summary_quality,improved_summary_quality,action_item_quality,keyword_quality,latency_seconds,notes
0,Clean audio,,,,,,,
1,Fast audio,,,,,,,
2,Noisy audio,,,,,,,
3,meeting-clip1,,,,,,,
4,meeting-clip2,,,,,,,
